# Demo 3: Spike-temporal Hypergraph (GSE + HIF)

Investor narrative: Adding time/topology — spike coincidence forms crisp hyperedges that reduce polysemanticity relative to earlier representations (see DEMO_CORRIDOR.md).


In [ ]:
import os, sys, json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch, transformers
print({
    'python': sys.version,
    'numpy': np.__version__,
    'matplotlib': matplotlib.__version__,
    'torch': torch.__version__,
    'transformers': transformers.__version__,
})

try:
    import py_nsi  # noqa: F401
    print('py_nsi import: OK')
except Exception as e:
    raise RuntimeError(
        'py_nsi is not importable. Build the local Rust wheel first:\n'
        '  cd py_nsi && maturin develop --release\n'
        'Then re-run this notebook.'
    ) from e

# Make local modules importable (namespace package "python" at repo root)
sys.path.append(os.path.abspath('.'))


In [ ]:
from python.utils.config import load_yaml
cfg = load_yaml('configs/demo3_spike_hypergraph.yaml')
cfg

In [ ]:
from python.datasets.bank_sentences import generate_bank_dataset
ds_cfg = cfg['dataset']
texts, labels = generate_bank_dataset(n_per_class=int(ds_cfg['n_per_class']), seed=int(ds_cfg['seed']))
labels_np = np.asarray(labels, dtype=np.int32)
num_concepts = int(len(set(labels)))
concept_names = ds_cfg.get('concepts', [f'concept_{i}' for i in range(num_concepts)])
print('Samples:', len(texts), ' Num concepts:', num_concepts)
print('Class counts:', {c:int((labels_np==c).sum()) for c in range(num_concepts)})
print('\nPreview:\n', '\n'.join(texts[:5]))


In [ ]:
from python.activations.extract import get_model_and_tokenizer, capture_layer_activations
model_name = cfg['model_name']
layer_index = int(cfg['layer_index'])
model, tokenizer = get_model_and_tokenizer(model_name)
acts = capture_layer_activations(model, tokenizer, texts, layer_index=layer_index)
acts.shape

In [ ]:
from python.ensemble.intersection import build_pyensemble
ens_cfg = cfg['ensemble']
input_dim = int(acts.shape[1])
os.environ['PY_NSI_INPUT_DIM'] = str(input_dim)
ensemble = build_pyensemble(feature_dim=int(ens_cfg['feature_dim']), top_k=int(ens_cfg['top_k']), seeds=[int(s) for s in ens_cfg['seeds']])
ensemble

In [ ]:
from python.hypergraph.pipeline import build_hypergraph
spk = cfg['spike']
store, features_bool, edge_keys = build_hypergraph(
    ensemble=ensemble,
    acts=acts,
    labels=labels_np,
    t_start=float(spk['t_start']),
    delta_t=float(spk['delta_t']),
    min_sigmoid=float(spk['min_sigmoid']),
    gse_window=float(spk['gse_window']),
)
features_bool.shape, len(edge_keys)

In [ ]:
from python.metrics.polysemanticity import concept_probs, poly_count, entropy, summarize_polysemanticity
met = cfg['metrics']
eps = float(met['eps'])
active_threshold_hyperedge = float(met['active_threshold_hyperedge'])
features_float = features_bool.astype(np.float32)
E = features_float.shape[1]
if E == 0:
    print('No hyperedges formed; adjust gse_window/min_sigmoid/top_k and re-run.')
    prob_h = np.zeros((0, num_concepts), dtype=np.float32)
    poly_h = np.zeros((0,), dtype=np.int32)
    ent_h = np.zeros((0,), dtype=np.float32)
else:
    prob_h = concept_probs(features_float, labels_np, num_concepts=num_concepts, active_threshold=active_threshold_hyperedge)
    poly_h = poly_count(prob_h, eps=eps)
    ent_h = entropy(prob_h)
summary = summarize_polysemanticity(prob_h, eps=eps) if E>0 else {'median_poly':0.0,'p90_poly':0.0,'monosemantic_rate':0.0}
summary, {'E': int(E)}


In [ ]:
from python.plots.hist import plot_histogram
if features_bool.shape[1] > 0:
    plt.figure(figsize=(6,4));
    plt.hist(poly_h, bins=int(met['hist_bins']), color='#4C78A8', edgecolor='white');
    plt.title(f"Spike-temporal Hyperedge poly (eps={eps:.2g}) — median={summary['median_poly']:.2f}, mono={summary['monosemantic_rate']:.1%}");
    plt.xlabel('poly(f)'); plt.ylabel('frequency'); plt.show();

    # Top-entropy hyperedges
    k = min(3, len(ent_h))
    top_ent_idx = np.argsort(-ent_h)[:k]
    print('\nTop-entropy hyperedges:')
    for idx in top_ent_idx:
        print('edge idx', int(idx), 'size', len(edge_keys[idx]), 'entropy', float(ent_h[idx]), 'probs', {concept_names[j]: float(prob_h[idx, j]) for j in range(num_concepts)})

    # Top-monosemantic (poly==1), choose highest max-prob
    mono_idx = np.where(poly_h == 1)[0]
    if len(mono_idx) > 0:
        mono_sorted = sorted(mono_idx, key=lambda i: float(prob_h[i].max()), reverse=True)[:k]
        print('\nTop-monosemantic hyperedges:')
        for idx in mono_sorted:
            winner = int(np.argmax(prob_h[idx]))
            print('edge idx', int(idx), 'size', len(edge_keys[idx]), 'winner', concept_names[winner], 'p=', float(prob_h[idx, winner]))
    else:
        print('\nNo monosemantic (poly==1) hyperedges found.')
else:
    print('Skipping plots/listing due to E==0.')


In [ ]:
from python.utils.artifacts import create_run_dir, dump_json, dump_yaml
met = cfg['metrics']; out = cfg['outputs']; spk = cfg['spike']
run_dir = create_run_dir(base_dir=out['base_dir'], run_tag=out.get('run_tag'))

# Save arrays and configs
np.save(os.path.join(run_dir, 'features_hyperedges.npy'), features_bool)
with open(os.path.join(run_dir, 'edge_keys.json'), 'w', encoding='utf-8') as f:
    json.dump([list(map(int, t)) for t in edge_keys], f, indent=2)
dump_yaml(cfg, os.path.join(run_dir, 'config.yaml'))

# Metrics JSON
E = int(features_bool.shape[1])
edge_sizes = [len(k) for k in edge_keys]
metrics_h = {
    'representation': 'hyperedges_spike_temporal',
    'num_features': int(prob_h.shape[0]) if E>0 else 0,
    'num_concepts': int(num_concepts),
    'concepts': list(concept_names),
    'num_samples': int(features_bool.shape[0]),
    'num_edges': int(E),
    'edge_size_median': float(np.median(edge_sizes)) if len(edge_sizes)>0 else 0.0,
    'edge_size_p90': float(np.percentile(edge_sizes, 90.0)) if len(edge_sizes)>0 else 0.0,
    'gse_window': float(spk['gse_window']),
    'spike': {'t_start': float(spk['t_start']), 'delta_t': float(spk['delta_t']), 'min_sigmoid': float(spk['min_sigmoid'])},
    'thresholds': {
        'eps': float(met['eps']),
        'active_threshold_hyperedge': float(met['active_threshold_hyperedge']),
    },
    'median_poly': float(np.median(poly_h)) if E>0 else 0.0,
    'p90_poly': float(np.percentile(poly_h, 90.0)) if E>0 else 0.0,
    'monosemantic_rate': float((poly_h==1).sum()/max(E,1)) if E>0 else 0.0,
}
from python.metrics.downstream import evaluate_logreg
acc = 0.0 if E==0 else float(evaluate_logreg(features_bool.astype(np.float32), labels_np, seed=int(ds_cfg['seed']))['accuracy'])
metrics_h['accuracy'] = acc
dump_json(metrics_h, os.path.join(run_dir, 'metrics_hyperedges.json'))

# HIF export
store.export_hif(os.path.join(run_dir, 'hypergraph.hif.json'))

# Plot to file (also display inline if E>0)
if E>0:
    from python.plots.hist import plot_histogram
    title_h = f"Spike-temporal Hyperedge polysemanticity (eps={met['eps']:.2g}) — median={metrics_h['median_poly']:.2f}, mono={metrics_h['monosemantic_rate']:.1%}"
    plot_histogram(poly_h, bins=int(met['hist_bins']), title=title_h, path=os.path.join(run_dir, 'poly_hist_hyperedges.png'))

print('Artifacts saved to:', run_dir)
print('Acceptance: HIF, metrics, features, edge_keys, config, histogram saved. See DEMO_CORRIDOR.md for narrative alignment.')
